In [ ]:
# @title 1. KẾT NỐI HỆ THỐNG { display-mode: "form" }
from google.colab import drive
import os
import requests
import pandas as pd
from google.colab import files
from tqdm.notebook import tqdm

print("🔄 Đang kết nối Google Drive...")
drive.mount('/content/drive', force_remount=True)
print("✅ Đã kết nối thành công.")

In [ ]:
# @title 2. CẤU HÌNH DOWNLOAD { display-mode: "form" }

# @markdown ### **Bước A: Chọn file CSV danh sách từ AWS**
print("📂 Vui lòng chọn file CSV:")
uploaded = files.upload()
if not uploaded:
    raise Exception("Chưa tải file CSV lên, vui lòng chạy lại ô này.")
csv_file = list(uploaded.keys())[0]
df = pd.read_csv(csv_file)

# @markdown ---
# @markdown ### **Bước B: Thiết lập nơi lưu trữ**

# @markdown **Chọn loại bộ nhớ:**
STORAGE_TYPE = "MyDrive" # @param ["MyDrive", "Shared Drives"]

# @markdown **Input folder path or folder ID**:
# @markdown (Example: /Downloads/TestABC or 1lzn2MQiBek6w4bIntHaGy3OFg2G9hSBj):
FOLDER_INPUT = "" # @param {type:"string"}

def get_final_path(storage_type, folder_input):
    base_path = "/content/drive/MyDrive" if storage_type == "MyDrive" else "/content/drive/Shared drives"

    # Trường hợp 1: User nhập ID (Chuỗi ký tự loằng ngoằng không chứa dấu gạch chéo)
    if len(folder_input) > 20 and "/" not in folder_input:
        print(f"🔍 Đang kiểm tra Folder ID: {folder_input}...")
        # Lưu ý: Với ID, ta sẽ dùng đường dẫn tạm để gdown hoặc xử lý,
        # nhưng cách an toàn nhất cho bác sĩ là yêu cầu nhập Tên hoặc Path tính từ gốc Drive.
        # Ở đây ta giả định bác sĩ nhập Tên thư mục cho đơn giản.
        return os.path.join(base_path, folder_input)

    # Trường hợp 2: User nhập tên thư mục hoặc đường dẫn con
    final_path = os.path.join(base_path, folder_input.lstrip('/'))
    return final_path

FINAL_SAVE_PATH = get_final_path(STORAGE_TYPE, FOLDER_INPUT)

# Tạo thư mục nếu chưa có
os.makedirs(FINAL_SAVE_PATH, exist_ok=True)

print(f"\n✅ Đã xác nhận nơi lưu:")
print(f"📍 Đường dẫn: {FINAL_SAVE_PATH}")
print(f"📊 Sẵn sàng tải: {len(df)} file.")

In [ ]:
# @title 3. BẮT ĐẦU TẢI DỮ LIỆU { display-mode: "form" }

def stream_aws_to_drive(url, filename, save_path):
    dest_file = os.path.join(save_path, filename)

    if os.path.exists(dest_file):
        print(f"⏩ Đã tồn tại: {filename} (Bỏ qua)")
        return

    # Stream từ AWS sang Drive (1MB mỗi lần)
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        total_size = int(r.headers.get('content-length', 0))

        with open(dest_file, 'wb') as f, tqdm(
            desc=filename,
            total=total_size,
            unit='B',
            unit_scale=True,
            unit_divisor=1024,
        ) as bar:
            for chunk in r.iter_content(chunk_size=1024*1024):
                f.write(chunk)
                bar.update(len(chunk))

# Vòng lặp thực thi
print(f"🚀 Tiến trình đang chạy... Đừng đóng trình duyệt!")
try:
    for index, row in df.iterrows():
        # Lấy link AWS và tên file từ CSV (hãy đảm bảo cột đúng tên 'url' và 'file_name')
        aws_url = row['url']
        file_name = row['file_name']

        stream_aws_to_drive(aws_url, file_name, FINAL_SAVE_PATH)

    print("\n" + "="*40)
    print("🎉 HOÀN THÀNH: Toàn bộ dữ liệu đã nằm an toàn trên Drive!")
    print(f"📁 Thư mục: {FINAL_SAVE_PATH}")
except Exception as e:
    print(f"\n❌ LỖI TRONG QUY TRÌNH: {e}")